**Loan Approval : CreditWise Loan System**

A mid-sized financial company named SecureTrust Bank
offers personal and home loans to customers across urban and rural regions of India. Every day,
hundreds of customers apply for loans through online and branch applications.

Until now, SecureTrust Bank has been using a manual verification process where loan officers evaluate applications by checking income proofs, employment
details, credit history, and other documents. This process is time-consuming, biased & inconsistent.

As a result, the bank faces two major challenges:

1.Good customers sometimes get rejected, leading to loss of business.

2.High-risk customers sometimes get approved
, leading to financial losses.

To solve this problem, the bank wants to introduce an system intelligent loan approval powered by Machine Learning that can automatically analyse applicant details and predict whether a loan should be Approved or Rejected human verification.



**This is a Binary Classfication Problem**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("/content/loan_approval_data.csv")

In [ ]:
df.head()
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

#Handeling Missing Values

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns
numerical_cols = df.select_dtypes(include=["number"]).columns

In [ ]:
categorical_cols

In [ ]:
numerical_cols

In [ ]:
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy="mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])

In [ ]:
cat_imp = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])

In [ ]:
df.head()

In [ ]:
df.head()
df.isnull().sum()

#Exploratory Data Analysis (EDA):

In [ ]:
#How Balanced the classes are?
classes_count = df["Loan_Approved"].value_counts()

plt.pie(classes_count, labels=["No", "Yes"], autopct="%1.1f%%")
plt.title("Is Loan approved or not?")

In [ ]:
#Analyze the Categories
#For Gender Count:
gender_count = df["Gender"].value_counts()
ax= sns.barplot(x=gender_count.index, y=gender_count.values)
ax.bar_label(ax.containers[0])
plt.title("Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.show()

In [ ]:
#For Edcuation:
education_count = df["Education_Level"].value_counts()
ax= sns.barplot(x=education_count.index, y=education_count.values)
ax.bar_label(ax.containers[0])
plt.title("Education Distribution")
plt.xlabel("Education")
plt.ylabel("Count")
plt.show()

In [ ]:
# Analyze Income for the Applicant

sns.histplot(
    data = df,
    x = "Applicant_Income",
    bins=20
)

In [ ]:
# Analyze Income for the Applicant

sns.histplot(
    data = df,
    x = "Coapplicant_Income",
    bins=20
)

In [ ]:
sns.histplot(df['Applicant_Income'], color='blue', bins=20, kde=True)
sns.histplot(df['Coapplicant_Income'], color='red', bins=20, kde=True)

In [ ]:
df['Coapplicant_Income_bin'] = pd.cut(df['Coapplicant_Income'], bins=3)

sns.histplot(
    data=df,
    x='Applicant_Income',
    hue='Coapplicant_Income_bin',
    bins=20
)

In [ ]:
#Outliers by BoxPlots:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="Loan_Approved",
    y="Applicant_Income"
)

plt.title("Applicant Income Distribution by Loan Approval Status")
plt.xlabel("Loan Approval (0 = No, 1 = Yes)")
plt.ylabel("Applicant Income")

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot 1
sns.boxplot(ax=axes[0, 0], data=df, x="Loan_Approved", y="Applicant_Income")
axes[0, 0].set_title("Applicant Income vs Loan Approval")
axes[0, 0].set_xlabel("Loan Approval")
axes[0, 0].set_ylabel("Applicant Income")

# Plot 2
sns.boxplot(ax=axes[0, 1], data=df, x="Loan_Approved", y="Credit_Score")
axes[0, 1].set_title("Credit Score vs Loan Approval")
axes[0, 1].set_xlabel("Loan Approval")
axes[0, 1].set_ylabel("Credit Score")

# Plot 3
sns.boxplot(ax=axes[1, 0], data=df, x="Loan_Approved", y="DTI_Ratio")
axes[1, 0].set_title("DTI Ratio vs Loan Approval")
axes[1, 0].set_xlabel("Loan Approval")
axes[1, 0].set_ylabel("DTI Ratio")

# Plot 4
sns.boxplot(ax=axes[1, 1], data=df, x="Loan_Approved", y="Savings")
axes[1, 1].set_title("Savings vs Loan Approval")
axes[1, 1].set_xlabel("Loan Approval")
axes[1, 1].set_ylabel("Savings")

plt.tight_layout()
plt.show()

In [ ]:
# Credit Score with Loan Approved

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="Loan_Approved", y="Credit_Score")


In [ ]:
#Credit Score with Loan Approved
sns.histplot(
    data=df,
    x="Credit_Score",
    hue="Loan_Approved",
    bins=20,
    multiple="dodge"
)

The above plot shows a clear relationship between credit score and loan approval. Applicants with higher credit scores **(around 700–800)** are more likely to have their loans approved, while those with lower scores (below 650) are mostly rejected. There is some overlap in the mid-range **(650–700)**, indicating that other factors like income or DTI ratio may also influence the decision. Overall, credit score appears to be a strong and important feature for predicting loan approval.

In [ ]:
# Remove Applicant Id
df = df.drop("Applicant_ID", axis=1)

#Feature Encoding:

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le = LabelEncoder()
df["Education_Level"] = le.fit_transform(df["Education_Level"])
df["Loan_Approved"] = le.fit_transform(df["Loan_Approved"])


In [ ]:
df.head()

In [ ]:
cols = ["Employment_Status", "Marital_Status", "Loan_Purpose", "Property_Area", "Gender", "Employer_Category"]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")

encoded = ohe.fit_transform(df[cols])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=df.index)

df = pd.concat([df.drop(columns=cols), encoded_df], axis=1)

In [ ]:
df.head()
df.info()

#Coorelation Heatmap:

In [ ]:
num_cols = df.select_dtypes(include="number")
corr_matrix = num_cols.corr()

plt.figure(figsize=(15, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)


In [ ]:
num_cols.corr()["Loan_Approved"].sort_values(ascending=False)

#Train-Test-Split + Feature Scaling

In [ ]:
X = df.drop("Loan_Approved", axis=1)
y = df["Loan_Approved"]

In [ ]:
y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_test.head()

In [ ]:
df["Coapplicant_Income_bin"] = pd.cut(
    df["Coapplicant_Income"],
    bins=3,
    labels=[0, 1, 2]
)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X = df.drop("Loan_Approved", axis=1)
y = df["Loan_Approved"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_test_scaled

#Train & Evaluate Models

#Logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score


In [ ]:
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)


In [ ]:
# Evaluation
print("Logistic Regression Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

#KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

In [ ]:
# Evaluation
print("KNN Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

#Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

In [ ]:
# Evaluation
print("Naive Bayes Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

**Result: Best Model on the basis of Precision => Naive Bayes**

#Feature Engineering

In [ ]:
# Add or Tranform features
df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2

# df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"])

X = df.drop(columns=["Loan_Approved", "Credit_Score", "DTI_Ratio"])
y = df["Loan_Approved"]

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train.head()

In [ ]:
# Logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

# Evaluation
print("Logistic Regression Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

In [ ]:
# KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation
print("KNN Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation
print("Naive Bayes Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))